# telemetry provenance / coverage check

before going further with telemetry I want to sort out what's actually ground truth vs what we happened to get telemetry for. incident_events.csv is the full incident log (69 incidents) regardless of whether telemetry exists for them. the telemetry side (manifest.csv + meta.json + the actual files) is just whatever subset got extracted - 21 files, 16 tied to one specific incident and 5 are the "node good" baseline quarters (not tied to an incident at all).

so if a category or node has no telemetry that doesn't mean it was excluded for a reason, it probably just was never pulled in the first place. keeping that distinction straight for the rest of this.

In [1]:
import json
import re
import sys
from pathlib import Path

# make the repo root importable so `src` works from any folder
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'paths.py').exists())
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.paths import DATA_RAW as DATA, REPORTS

incidents = pd.read_csv(DATA / 'incident_events.csv')
manifest = pd.read_csv(DATA / 'manifest.csv')

print('incident_events.csv rows:', len(incidents))
print('manifest.csv rows:', len(manifest))

incident_events.csv rows: 69
manifest.csv rows: 21


## 1. join key

incident_events.csv identifies incidents by (node, incidentDate). manifest.csv's inputFile name has basically the same info baked in: `<hostcode>_<date>_<category-slug>.csv.bz2`, plus it has a node column directly. so (node, date) is the obvious join key. dropping the 5 node-wide-totals files here since they're not tied to a single incident.

In [2]:
incidents = incidents.copy()
incidents['incidentDateParsed'] = pd.to_datetime(incidents['incidentDate'], format='%d %b %Y').dt.date
incidents['incidentId'] = incidents['node'] + '_' + incidents['incidentDateParsed'].astype(str)

is_baseline = manifest['inputType'] == 'node-wide-totals'
baseline_manifest = manifest[is_baseline].copy()
incident_manifest = manifest[~is_baseline].copy()

FILE_PATTERN = re.compile(r'^(?P<host>[a-z0-9]+)_(?P<date>\d{4}-\d{2}-\d{2})_(?P<slug>.+)\.csv\.bz2$')

parsed = incident_manifest['inputFile'].str.extract(FILE_PATTERN)
incident_manifest['hostCode'] = parsed['host']
incident_manifest['fileDate'] = pd.to_datetime(parsed['date']).dt.date
incident_manifest['categorySlug'] = parsed['slug']
incident_manifest['telemetryId'] = incident_manifest['node'] + '_' + incident_manifest['fileDate'].astype(str)

incident_manifest[['inputFile', 'node', 'fileDate', 'categorySlug', 'telemetryId']]

,inputFile,node,fileDate,categorySlug,telemetryId
0,ggpu121_2025-02-10_gpu-error.csv.bz2,node_2b08d249c0ba,2025-02-10,gpu-error,node_2b08d249c0ba_2025-02-10
1,ggpu129_2026-01-09_gpu-lost.csv.bz2,node_b1b91a387252,2026-01-09,gpu-lost,node_b1b91a387252_2026-01-09
2,ggpu129_2026-01-13_gpu-lost.csv.bz2,node_b1b91a387252,2026-01-13,gpu-lost,node_b1b91a387252_2026-01-13
3,ggpu129_2026-01-19_gpu-lost.csv.bz2,node_b1b91a387252,2026-01-19,gpu-lost,node_b1b91a387252_2026-01-19
4,ggpu139_2025-03-21_gpus-dropped-off-bus-maybe....,node_df3724dfcace,2025-03-21,gpus-dropped-off-bus-maybe,node_df3724dfcace_2025-03-21
5,ggpu142_2025-02-17_gpus-fallen-off-bus.csv.bz2,node_6a5ebc855306,2025-02-17,gpus-fallen-off-bus,node_6a5ebc855306_2025-02-17
6,ggpu142_2025-03-15_unknown-error-possibly-nong...,node_6a5ebc855306,2025-03-15,unknown-error-possibly-nongpu,node_6a5ebc855306_2025-03-15
7,ggpu142_2025-03-19_gpu-problem.csv.bz2,node_6a5ebc855306,2025-03-19,gpu-problem,node_6a5ebc855306_2025-03-19
8,ggpu142_2025-03-21_gpus-fell-off-bus.csv.bz2,node_6a5ebc855306,2025-03-21,gpus-fell-off-bus,node_6a5ebc855306_2025-03-21
9,ggpu142_2025-07-03_gpu-problem.csv.bz2,node_6a5ebc855306,2025-07-03,gpu-problem,node_6a5ebc855306_2025-07-03


## 2. bring meta.json into the chain too

manifest.csv = what got extracted. meta.json = how each file got processed (kept metrics, schema, etc). joining it in so we have the full chain incident_events -> manifest -> meta.json -> file, and checking the node id agrees between manifest and meta as a sanity check.

In [3]:
def load_meta(output_file: str) -> pd.Series:
    meta_path = DATA / 'metadata' / output_file.replace('_tidy.csv.bz2', '_meta.json')
    if not meta_path.exists():
        return pd.Series({'metaNode': None, 'metaKeptMetricCount': None, 'metaExists': False})
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    return pd.Series({
        'metaNode': meta.get('node'),
        'metaKeptMetricCount': len(meta.get('keptMetrics', [])),
        'metaExists': True,
    })

incident_manifest = incident_manifest.join(incident_manifest['outputFile'].apply(load_meta))

assert incident_manifest['metaExists'].all(), 'missing meta.json for some manifest rows'
assert (incident_manifest['node'] == incident_manifest['metaNode']).all(), 'node mismatch between manifest.csv and meta.json'
print('Provenance chain intact for all', len(incident_manifest), 'incident-scoped telemetry files: manifest.node == meta.json.node in every case.')

Provenance chain intact for all 16 incident-scoped telemetry files: manifest.node == meta.json.node in every case.


## 3. audit table

outer join on (node, date), then label each row:
- telemetry available - incident has a matching file
- telemetry unavailable - incident is logged but we never got telemetry for it
- unmatched telemetry - a file exists but doesn't match any logged incident

In [4]:
audit = incidents.merge(
    incident_manifest,
    left_on='incidentId',
    right_on='telemetryId',
    how='outer',
    suffixes=('_incident', '_manifest'),
    indicator=True,
)

status_map = {'both': 'telemetry_available', 'left_only': 'telemetry_unavailable', 'right_only': 'unmatched_telemetry'}
audit['status'] = audit['_merge'].map(status_map)

audit_out = audit[[
    'node_incident', 'node_manifest', 'incidentDate', 'incidentDateParsed', 'category',
    'fileDate', 'categorySlug', 'inputFile', 'outputFile', 'status',
]].rename(columns={'node_incident': 'node_from_incident_log', 'node_manifest': 'node_from_manifest'})

audit_out.to_csv(REPORTS / 'incident_telemetry_audit.csv', index=False)
print(audit_out['status'].value_counts())
audit_out.sort_values(['status', 'node_from_incident_log'], na_position='last')

status
telemetry_unavailable    54
telemetry_available      15
unmatched_telemetry       1
Name: count, dtype: int64


,node_from_incident_log,node_from_manifest,incidentDate,incidentDateParsed,category,fileDate,categorySlug,inputFile,outputFile,status
3,node_0876e5fac0c8,NaN,22 Oct 2025,2025-10-22,gpu fell off bus,NaN,NaN,NaN,NaN,telemetry_unavailable
4,node_15e3440a80d0,NaN,07 Jan 2025,2025-01-07,gpu ecc,NaN,NaN,NaN,NaN,telemetry_unavailable
5,node_15e3440a80d0,NaN,14 Jan 2025,2025-01-14,gpu ecc,NaN,NaN,NaN,NaN,telemetry_unavailable
6,node_1a061e79b6d1,NaN,12 Jun 2025,2025-06-12,gpu error/problem,NaN,NaN,NaN,NaN,telemetry_unavailable
7,node_1a061e79b6d1,NaN,12 Sep 2025,2025-09-12,gpu handle error,NaN,NaN,NaN,NaN,telemetry_unavailable
...,...,...,...,...,...,...,...,...,...,...
57,node_b1b91a387252,node_b1b91a387252,09 Jan 2026,2026-01-09,gpu lost,2026-01-09,gpu-lost,ggpu129_2026-01-09_gpu-lost.csv.bz2,ggpu129_2026-01-09_gpu-lost_tidy.csv.bz2,telemetry_available
58,node_b1b91a387252,node_b1b91a387252,13 Jan 2026,2026-01-13,gpu lost,2026-01-13,gpu-lost,ggpu129_2026-01-13_gpu-lost.csv.bz2,ggpu129_2026-01-13_gpu-lost_tidy.csv.bz2,telemetry_available
59,node_b1b91a387252,node_b1b91a387252,19 Jan 2026,2026-01-19,gpu lost,2026-01-19,gpu-lost,ggpu129_2026-01-19_gpu-lost.csv.bz2,ggpu129_2026-01-19_gpu-lost_tidy.csv.bz2,telemetry_available
62,node_d4119f476626,node_d4119f476626,21 Mar 2025,2025-03-21,gpu error/problem,2025-03-21,gpu-problem,ggpu144_2025-03-21_gpu-problem.csv.bz2,ggpu144_2025-03-21_gpu-problem_tidy.csv.bz2,telemetry_available


15 available / 54 unavailable out of the 69 incidents. of the 16 incident-scoped files, 15 matched and 1 didn't. saved the table to reports/incident_telemetry_audit.csv so I don't have to redo this join every time.

## 4. coverage by node, category, incident frequency

In [5]:
incidents['hasTelemetry'] = incidents['incidentId'].isin(incident_manifest['telemetryId'])

coverage_by_category = (
    incidents.groupby('category')['hasTelemetry']
    .agg(incidentsWithTelemetry='sum', totalIncidents='count')
    .assign(coverageRate=lambda d: d['incidentsWithTelemetry'] / d['totalIncidents'])
    .sort_values('totalIncidents', ascending=False)
)
coverage_by_category

,incidentsWithTelemetry,totalIncidents,coverageRate
category,,,
gpu error/problem,5,31,0.161290
gpu fell off bus,7,24,0.291667
gpu unknown,0,5,0.000000
gpu lost,3,3,1.000000
gpu failed,0,2,0.000000
gpu ecc,0,2,0.000000
gpu handle error,0,1,0.000000
gpu timeout,0,1,0.000000


In [6]:
coverage_by_node = (
    incidents.groupby('node')['hasTelemetry']
    .agg(incidentsWithTelemetry='sum', totalIncidents='count')
    .assign(coverageRate=lambda d: d['incidentsWithTelemetry'] / d['totalIncidents'])
    .sort_values('totalIncidents', ascending=False)
)
coverage_by_node

,incidentsWithTelemetry,totalIncidents,coverageRate
node,,,
node_ada68b80182e,0,15,0.0
node_95799a17101c,0,11,0.0
node_2348d3608e24,0,5,0.0
node_6a5ebc855306,5,5,1.0
node_1a061e79b6d1,0,3,0.0
node_dfc60a085f94,0,3,0.0
node_b1b91a387252,3,3,1.0
node_039ab2df8a6a,3,3,1.0
node_484a8d060485,0,2,0.0


coverage doesn't track incident frequency at all - it's basically all-or-nothing per node. node_ada68b80182e (15 incidents!) and node_95799a17101c (11 incidents) have zero telemetry. meanwhile every node that has any telemetry has ALL of its logged incidents covered (rate = 1.0 every time). looks like whoever extracted this picked a fixed set of nodes rather than sampling incidents one by one - but that's just my read of the pattern, manifest/meta don't actually say how nodes were chosen. need to ask about this rather than assume it.

In [7]:
freq_bins = [0, 1, 3, 6, 100]
freq_labels = ['1 incident', '2-3 incidents', '4-6 incidents', '7+ incidents']
node_freq = incidents.groupby('node')['hasTelemetry'].agg(incidentsWithTelemetry='sum', totalIncidents='count')
node_freq['freqBucket'] = pd.cut(node_freq['totalIncidents'], bins=freq_bins, labels=freq_labels)

coverage_by_frequency = (
    node_freq.groupby('freqBucket', observed=True)
    .agg(nodesInBucket=('totalIncidents', 'size'), nodesWithAnyTelemetry=('incidentsWithTelemetry', lambda s: (s > 0).sum()))
)
coverage_by_frequency

,nodesInBucket,nodesWithAnyTelemetry
freqBucket,,
1 incident,15,4
2-3 incidents,7,2
4-6 incidents,2,1
7+ incidents,2,0


## 5. the ggpu142 mismatch

ggpu142 has 6 telemetry files but only 5 incidents logged for its node (node_6a5ebc855306). 2025-03-15_unknown-error-possibly-nongpu is the extra one - digging into why it doesn't match below.

In [8]:
node_id = 'node_6a5ebc855306'

print('--- incidents logged for', node_id, '---')
display(incidents.loc[incidents['node'] == node_id, ['node', 'incidentDate', 'incidentDateParsed', 'category']])

print('--- telemetry files we have for', node_id, '---')
display(incident_manifest.loc[incident_manifest['node'] == node_id, ['inputFile', 'node', 'fileDate', 'categorySlug']])

unmatched = incident_manifest.loc[incident_manifest['telemetryId'] == f'{node_id}_2025-03-15']
print('--- checking the 2025-03-15 file field by field ---')
print('node:     manifest says', unmatched['node'].iloc[0], ', meta.json says', unmatched['metaNode'].iloc[0], '- these agree, and this node does have incidents logged')
print('date:     file is 2025-03-15, incident dates for this node are 2025-02-17 / 2025-03-19 / 2025-03-21 / 2025-07-03 / 2025-09-12 -> no match')
print('category: file slug is', unmatched['categorySlug'].iloc[0], ', incident categories for this node are gpu fell off bus / gpu error/problem -> doesn\'t match either')
print()
print('so node matches but date and category don\'t. keeping this one as unmatched rather than folding it into the 69 incidents.')
print('worth asking where this file actually came from - maybe pulled from raw logs outside incident_events.csv, or incident_events.csv just never logged it.')

--- incidents logged for node_6a5ebc855306 ---


,node,incidentDate,incidentDateParsed,category
11,node_6a5ebc855306,17 Feb 2025,2025-02-17,gpu fell off bus
12,node_6a5ebc855306,19 Mar 2025,2025-03-19,gpu error/problem
13,node_6a5ebc855306,21 Mar 2025,2025-03-21,gpu fell off bus
14,node_6a5ebc855306,03 Jul 2025,2025-07-03,gpu error/problem
15,node_6a5ebc855306,12 Sep 2025,2025-09-12,gpu fell off bus


--- telemetry files we have for node_6a5ebc855306 ---


,inputFile,node,fileDate,categorySlug
5,ggpu142_2025-02-17_gpus-fallen-off-bus.csv.bz2,node_6a5ebc855306,2025-02-17,gpus-fallen-off-bus
6,ggpu142_2025-03-15_unknown-error-possibly-nong...,node_6a5ebc855306,2025-03-15,unknown-error-possibly-nongpu
7,ggpu142_2025-03-19_gpu-problem.csv.bz2,node_6a5ebc855306,2025-03-19,gpu-problem
8,ggpu142_2025-03-21_gpus-fell-off-bus.csv.bz2,node_6a5ebc855306,2025-03-21,gpus-fell-off-bus
9,ggpu142_2025-07-03_gpu-problem.csv.bz2,node_6a5ebc855306,2025-07-03,gpu-problem
10,ggpu142_2025-09-12_gpus-dropped-off-bus.csv.bz2,node_6a5ebc855306,2025-09-12,gpus-dropped-off-bus


--- checking the 2025-03-15 file field by field ---
node:     manifest says node_6a5ebc855306 , meta.json says node_6a5ebc855306 - these agree, and this node does have incidents logged
date:     file is 2025-03-15, incident dates for this node are 2025-02-17 / 2025-03-19 / 2025-03-21 / 2025-07-03 / 2025-09-12 -> no match
category: file slug is unknown-error-possibly-nongpu , incident categories for this node are gpu fell off bus / gpu error/problem -> doesn't match either

so node matches but date and category don't. keeping this one as unmatched rather than folding it into the 69 incidents.
worth asking where this file actually came from - maybe pulled from raw logs outside incident_events.csv, or incident_events.csv just never logged it.


## summary

- incident_events.csv is ground truth, 69 incidents. telemetry is 21 files (16 incident-scoped + 5 baseline quarters).
- 15/69 incidents have telemetry, 54 don't. the ggpu142 2025-03-15 file doesn't match anything and stays flagged as unmatched, not counted as a 70th incident.
- coverage is per-node all-or-nothing, not proportional to how often a node fails. gpu unknown having 0 telemetry is just missing data, not something excluded on purpose.
- treat the 21 files as a biased subset, not a representative sample, until we actually know how they were picked.